<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/03_cross_dataset_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Phase 3: Cross-Dataset Evaluation (Generalization / Drift)
Evaluates both previously trained models — **no retraining** — on a second,
independent dataset built from real face-swap deepfake videos (a different
generation method than the StyleGAN images used in `01_baseline.ipynb`).

Requires `01_baseline.ipynb` and `02_finetuning.ipynb` to have been run at least
once, so `best_model.pth` and `best_model_finetuned.pth` already exist in your Drive.

Runtime → Change runtime type → GPU, then run cells top to bottom.
Note: this notebook does NOT need the original 140k-real-and-fake-faces dataset at all.

### Step 1 — Confirm you have a GPU

In [ ]:
!nvidia-smi

### Step 2 — Mount Google Drive

Connects your Drive so checkpoints can be saved/loaded from
`My Drive/thesis_checkpoints`, surviving across separate Colab sessions
and separate notebooks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to / loaded from:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be saved to / loaded from: /content/drive/MyDrive/thesis_checkpoints


### Step 3 — Kaggle API access

Upload your `kaggle.json` (from https://www.kaggle.com/settings/account →
Create New Token). Only needed once per Colab session.

In [ ]:
import os

KAGGLE_API_TOKEN = os.getenv("KGAT_305e81603250999b12324cc02e529f89")


### Step 4 — Download the second, independent dataset

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip -q deepfake-and-real-images.zip -d data_raw_2
!find data_raw_2 -maxdepth 4 -type d

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
deepfake-and-real-images.zip: Skipping, found more recently modified local copy (use --force to force download)
replace data_raw_2/Dataset/Test/Fake/fake_0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
data_raw_2
data_raw_2/Dataset
data_raw_2/Dataset/Test
data_raw_2/Dataset/Test/Real
data_raw_2/Dataset/Test/Fake
data_raw_2/Dataset/Train
data_raw_2/Dataset/Train/Fake


### Step 5 — Auto-locate the test split

Searches for a folder containing `Real`/`real` and `Fake`/`fake` subfolders,
preferring one with "test" in its path.

In [ ]:
import os

def find_real_fake_folders(root):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        lower = [d.lower() for d in dirnames]
        if "real" in lower and "fake" in lower:
            candidates.append(dirpath)
    return candidates

candidates = find_real_fake_folders("data_raw_2")
print("Found candidate folders:")
for c in candidates:
    print(" -", c)

test_candidates = [c for c in candidates if "test" in c.lower()]
CROSS_TEST_DIR = test_candidates[0] if test_candidates else candidates[0]
print("\nUsing as cross-dataset test set:", CROSS_TEST_DIR)

Found candidate folders:
 - data_raw_2/Dataset/Test

Using as cross-dataset test set: data_raw_2/Dataset/Test


### Step 6 — Evaluate both models on this new, unseen data

Class names are matched case-insensitively against each checkpoint's own
`class_to_idx`, so the comparison is correct regardless of folder naming
differences between datasets.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])

cross_set = datasets.ImageFolder(CROSS_TEST_DIR, transform=transform)
cross_loader = DataLoader(cross_set, batch_size=64, shuffle=False, num_workers=2)
print("Cross-dataset class mapping (as found on disk):", cross_set.class_to_idx)
print("Cross-dataset size:", len(cross_set))


def evaluate_on_cross_dataset(checkpoint_path, model_name):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    ckpt_class_to_idx = checkpoint["class_to_idx"]

    remap = {}
    for name, idx_on_disk in cross_set.class_to_idx.items():
        key = name.lower()
        if key not in ckpt_class_to_idx:
            raise ValueError(f"Unexpected class folder name '{name}'")
        remap[idx_on_disk] = ckpt_class_to_idx[key]

    real_idx = ckpt_class_to_idx["real"]

    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()

    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in cross_loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, real_idx]
            preds = logits.argmax(dim=1)
            remapped_labels = [remap[l.item()] for l in labels]
            all_labels.extend(remapped_labels)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n=== {model_name} — Cross-Dataset (unseen source) ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"AUC: {auc:.4f}")
    print("Confusion matrix:\n", cm)
    return acc, auc


baseline_cross_acc, baseline_cross_auc = evaluate_on_cross_dataset(
    f"{CHECKPOINT_DIR}/best_model.pth", "Baseline #1 (frozen backbone)"
)
finetuned_cross_acc, finetuned_cross_auc = evaluate_on_cross_dataset(
    f"{CHECKPOINT_DIR}/best_model_finetuned.pth", "Phase 2 (fine-tuned layer4)"
)

print("\n\n=== SUMMARY: In-Distribution vs Cross-Dataset (Drift) ===")
print(f"{'Model':<30} {'In-dist Acc':<14} {'In-dist AUC':<14} {'Cross Acc':<12} {'Cross AUC':<12}")
print(f"{'Baseline #1':<30} {'0.8077':<14} {'0.8883':<14} {baseline_cross_acc:<12.4f} {baseline_cross_auc:<12.4f}")
print(f"{'Phase 2 (fine-tuned)':<30} {'fill in':<14} {'fill in':<14} {finetuned_cross_acc:<12.4f} {finetuned_cross_auc:<12.4f}")

Cross-dataset class mapping (as found on disk): {'Fake': 0, 'Real': 1}
Cross-dataset size: 10905

=== Baseline #1 (frozen backbone) — Cross-Dataset (unseen source) ===
Accuracy: 0.4868
AUC: 0.4887
Confusion matrix:
 [[ 289 5203]
 [ 393 5020]]

=== Phase 2 (fine-tuned layer4) — Cross-Dataset (unseen source) ===
Accuracy: 0.4945
AUC: 0.5253
Confusion matrix:
 [[  99 5393]
 [ 120 5293]]


=== SUMMARY: In-Distribution vs Cross-Dataset (Drift) ===
Model                          In-dist Acc    In-dist AUC    Cross Acc    Cross AUC   
Baseline #1                    0.8077         0.8883         0.4868       0.4887      
Phase 2 (fine-tuned)           fill in        fill in        0.4945       0.5253      


---
### Save for your report
Cross-dataset class mapping, accuracy/AUC for both models, and the summary table —
this is your generalization/drift result. A large drop is a normal, expected, and
legitimate finding at this stage, not a failure.

**Next: ensembling (ResNet18 + EfficientNet/ViT) and adversarial robustness.**